In [73]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from typing import Optional, Union
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm

SHAP_DATA = "/Users/jwhite22/Desktop/marmoset-paper/modeling/results/feature_analysis/b_tp4_TP4_MeanHU_LoeweFIC90_cholesterol_Constant_20250612_shap_data.csv"
IN_VITRO_DATA = "/Users/jwhite22/Desktop/marmoset-paper/data/in_vitro_diamond_data.csv"
SEED = 42

In [74]:
COMPOUND_COLORS = {
    'BDQ+DEL': '#4E79A7',
    'BDQ+LIN': '#F28E2B',
    'BDQ+PRE': '#E15759',
    'INH+PZA': '#76B7B2',
    'LIN+PRE': '#59A14F',
    'MOX+RIF': '#EDC948',
    'PZA+RIF': '#B07AA1',
    'BDQ+LIN+PRE': '#FF9DA7',
    'EMB+INH+PZA+RIF': '#9C755F',
    'EMB+MOX+PZA+RIF': '#BAB0AC',
}

COMPOUND_NAMES = {
    'BDQ+DEL': "BD",
    'BDQ+LIN': "BL",
    'BDQ+PRE': "BPa",
    'INH+PZA': "HZ",
    'LIN+PRE': "PaL",
    'MOX+RIF': "MR",
    'PZA+RIF': "PR",
    'BDQ+LIN+PRE': "BPaL",
    'EMB+INH+PZA+RIF': "HZRE",
    'EMB+MOX+PZA+RIF': "MRZE",
}

In [75]:
df_shap = pd.read_csv(SHAP_DATA)
df_invitro = pd.read_csv(IN_VITRO_DATA)

def merge_data(
    shap_df: pd.DataFrame,
    in_vitro_df: pd.DataFrame,
    drug_col: str = 'Drug',
    feature_col: str = 'feature_name',
    value_col: str = 'feature_value'
) -> pd.DataFrame:
    if drug_col not in in_vitro_df.columns:
        raise KeyError(f"'{drug_col}' not found in in vitro DataFrame")

    melt_df = in_vitro_df.melt(
        id_vars=[drug_col],
        var_name=feature_col,
        value_name=value_col
    )
    return shap_df.merge(melt_df, on=[feature_col, value_col], how='left')

In [76]:
# Merge datasets
merged = merge_data(df_shap, df_invitro)

# Generate and store jitter once for consistent positioning
np.random.seed(SEED)
merged['jitter'] = np.random.normal(loc=0.0, scale=0.08, size=len(merged))

# Create a modified 'vlag' colormap with a light grey center
#_base = plt.get_cmap('vlag')(np.linspace(0, 1, 256))
#_center = np.array([0.85, 0.85, 0.85, 1.0])
#_base[128] = _center
#VLG = LinearSegmentedColormap.from_list('vlag_grey_center', _base)

In [77]:
# Define Blue-Purple-Red colormap (positions normalized 0-1)
BPR = LinearSegmentedColormap.from_list(
    'blue_purple_red',
    [(0.0, '#4281C3'), (0.5, '#8C4199'), (1.0, '#ED1A55')]
)


In [87]:
# Plotting function using pre-computed jitter
def plot_shap_feature(
    df: pd.DataFrame,
    feature_name: str,
    color_by_value: bool = True,
    highlight_compound: Optional[str] = None,
    save_path: Optional[Union[str, Path]] = None,
    save_format: str = 'svg',
    dpi: int = 300
) -> None:
    plot_df = df[df['feature_name'] == feature_name].copy()
    jitter = plot_df['jitter']

    plt.figure(figsize=(8, 2))
    if color_by_value:
        # normalize feature values around median
        vmin = plot_df['feature_value'].min()
        vmax = plot_df['feature_value'].max()
        vcenter = np.median(plot_df['feature_value'])
        norm = TwoSlopeNorm(vmin=vmin, vcenter=vcenter, vmax=vmax)

        sc = plt.scatter(
            plot_df['shap_value'], jitter,
            c=plot_df['feature_value'], cmap=BPR, norm=norm,
            alpha=0.9, s=30
        )
        plt.colorbar(sc, label=feature_name)
    else:
        plt.scatter(
            plot_df['shap_value'], jitter,
            color='lightgrey', alpha=0.6, s=50
        )
        if highlight_compound:
            mask = plot_df['Drug'] == highlight_compound
            color = COMPOUND_COLORS.get(highlight_compound, 'black')
            name = COMPOUND_NAMES.get(highlight_compound, highlight_compound)
            plt.scatter(
                plot_df.loc[mask, 'shap_value'], jitter[mask],
                facecolors=color, edgecolors='black', s=50,
                linewidths=1.5, label=name
            )
            plt.legend()

    plt.xlabel('SHAP value')
    plt.yticks([])
    plt.title(f"SHAP for {feature_name}")
    plt.axvline(0, color='black', linestyle='--', linewidth=1)
    plt.tight_layout()

    if save_path:
        out = Path(save_path)
        if out.is_dir():
            fname = f"{feature_name}"
            if highlight_compound:
                fname += f"{highlight_compound}_SHAP"
            else:
                fname += "SHAP"
            fname += f".{save_format}"
            filepath = out / fname
        else:
            filepath = out 
        plt.savefig(filepath, dpi=dpi, bbox_inches='tight', format=save_format)
        plt.close()
        print(f"Saved plot to {filepath}")
    else:
        plt.show()

In [89]:
plot_shap_feature(merged, feature_name="LoeweFIC90_cholesterol_Constant", color_by_value=True, save_path=Path("figures/shap_highlights/"))
for compound in merged['Drug'].unique():
    plot_shap_feature(
        merged,
        feature_name="LoeweFIC90_cholesterol_Constant",
        color_by_value=False,
        save_path=Path("figures/shap_highlights/"),
        highlight_compound=compound
    )

Saved plot to figures/shap_highlights/LoeweFIC90_cholesterol_ConstantSHAP.svg
Saved plot to figures/shap_highlights/LoeweFIC90_cholesterol_ConstantBDQ+DEL_SHAP.svg
Saved plot to figures/shap_highlights/LoeweFIC90_cholesterol_ConstantBDQ+LIN_SHAP.svg
Saved plot to figures/shap_highlights/LoeweFIC90_cholesterol_ConstantBDQ+LIN+PRE_SHAP.svg
Saved plot to figures/shap_highlights/LoeweFIC90_cholesterol_ConstantBDQ+PRE_SHAP.svg
Saved plot to figures/shap_highlights/LoeweFIC90_cholesterol_ConstantEMB+INH+PZA+RIF_SHAP.svg
Saved plot to figures/shap_highlights/LoeweFIC90_cholesterol_ConstantEMB+MOX+PZA+RIF_SHAP.svg
Saved plot to figures/shap_highlights/LoeweFIC90_cholesterol_ConstantINH+PZA_SHAP.svg
Saved plot to figures/shap_highlights/LoeweFIC90_cholesterol_ConstantLIN+PRE_SHAP.svg
Saved plot to figures/shap_highlights/LoeweFIC90_cholesterol_ConstantMOX+RIF_SHAP.svg
Saved plot to figures/shap_highlights/LoeweFIC90_cholesterol_ConstantPZA+RIF_SHAP.svg


In [ ]:
# Single feature stacked plot for all compounds with grey background
def plot_shap_stacked(
    df: pd.DataFrame,
    feature_name: str,
    compounds: Optional[list[str]] = None,
    save_path: Optional[Union[str, Path]] = None,
    save_format: str = 'svg',
    dpi: int = 300
) -> None:
    # Filter for feature
    plot_df = df[df['feature_name'] == feature_name].copy()
    unique_drugs = compounds or plot_df['Drug'].unique().tolist()
    n = len(unique_drugs)

    # Setup figure
    fig, axes = plt.subplots(nrows=n, sharex=True, figsize=(8, 2 * n))
    if n == 1:
        axes = [axes]

    # Determine common x-limits
    xmin = plot_df['shap_value'].min()
    xmax = plot_df['shap_value'].max()

    # Plot each drug
    for ax, drug in zip(axes, unique_drugs):
        name = COMPOUND_NAMES.get(drug, drug)
        # background all points in grey
        ax.scatter(
            plot_df['shap_value'], plot_df['jitter'],
            color='lightgrey', alpha=0.6, s=30
        )
        # overlay compound-specific points
        sub = plot_df[plot_df['Drug'] == drug]
        ax.scatter(
            sub['shap_value'], sub['jitter'],
            facecolors=COMPOUND_COLORS.get(drug, 'black'),
            edgecolors=COMPOUND_COLORS.get(drug, 'black'),
            s=40, 
            alpha=0.9, 
            label=name
        )
        # formatting
        ax.set_yticks([])
        ax.set_ylabel(name, rotation=0, labelpad=40, va='center')
        ax.axvline(0, color='black', linestyle='--', linewidth=1)
        ax.set_xlim(xmin, xmax)

    # Common labels and title
    axes[-1].set_xlabel('SHAP value')
    fig.suptitle(f"Stacked SHAP for {feature_name}")
    plt.tight_layout(rect=[0, 0, 1, 0.96])

    # Save or show
    if save_path:
        out = Path(save_path)
        if out.is_dir():
            fname = f"{feature_name}_stacked.{save_format}"
            filepath = out / fname
        else:
            filepath = out
        plt.savefig(filepath, dpi=dpi, bbox_inches='tight', format=save_format)
        plt.close()
        print(f"Saved stacked plot to {filepath}")
    else:
        plt.show()

# Example usage: save as SVG
plot_shap_stacked(
    merged,
    feature_name="LoeweFIC90_cholesterol_Constant",
    save_path="figures/shap_stacked.svg",
    save_format='svg'
)


Saved stacked plot to figures/shap_stacked.svg
